# Please note, this Jupyter notebook won't successfully run by itself as it needs all the data from the zip file attached. If you need to validate result, please download the zip file with the attached instructions and datasets.

# Dependency

In [1]:
import pandas as pd
import numpy as np
import gurobipy as gp
from scipy.stats import truncnorm
from gurobipy import GRB
import json
import time
from helper import print_routes
from helper import display_route_map
from helper import reconstruct_route
from helper import display_mixed_route_map

# Data

### Constant setup

In [2]:
random_number_generator = np.random.default_rng(42)

### Distance
##### Store and residences distances: N+1 X N+1 matrix that represents distance between residences and store

In [3]:
with open("data/matrix.json") as f:
    matrix_data = json.load(f)
input_coord = matrix_data["coords"]
store_location = tuple(input_coord[0])
store_and_residences_distance_matrix = pd.DataFrame(matrix_data["distances_km"])
coordinates = []
for c in input_coord:
    coordinates.append(tuple(c))
route_geometries = matrix_data["routes"]

### Customer

In [4]:
number_of_customers = len(matrix_data["coords"]) - 1 # store at index 0, residences at 1..n
total_nodes = number_of_customers + 1
customers = range(1, total_nodes)
print(f"Total customer: {len(customers)}")

Total customer: 60


### Service Time
##### Service time in minutes by OSM building type. Apartments are slowest (buzzer + lift + unit-finding); terrace/semi-detached slightly longer than house due to narrower street access. Depot has zero service time. Residential is the fall back value for residential building that can't be classified

In [5]:
service_time_by_type = {
    "store":              0,
    "house":              5,
    "detached":           5,
    "bungalow":           5,
    "residential":        5,
    "terrace":            6,
    "semidetached_house": 6,
    "apartments":         15
}
service_times = [service_time_by_type[t] for t in matrix_data["types"]]
service_times

[0,
 15,
 6,
 5,
 15,
 5,
 5,
 5,
 5,
 5,
 5,
 6,
 5,
 5,
 5,
 15,
 15,
 15,
 5,
 5,
 15,
 15,
 5,
 5,
 15,
 5,
 15,
 15,
 6,
 15,
 5,
 5,
 5,
 5,
 5,
 5,
 15,
 6,
 15,
 5,
 15,
 5,
 5,
 5,
 5,
 15,
 5,
 5,
 5,
 15,
 5,
 5,
 5,
 5,
 15,
 5,
 5,
 5,
 5,
 5,
 5]

### Time Windows
##### Time windows: Four 2-hour slots (11:00–19:00), evening-weighted (10%/10%/30%/50%). Following the slot structure from woolworths.com.au with evening preference tailored for working households.

In [6]:
number_of_windows = 4
window_time = 2 * 60
total_time = number_of_windows * window_time
windows_allocation = [0.1, 0.1, 0.3, 0.5]
time_windows_mapping = ["11-13", "13-15", "15-17", "17-19"]
windows = [(i * window_time, (i + 1) * window_time) for i in range(number_of_windows)]
windows

[(0, 120), (120, 240), (240, 360), (360, 480)]

### Order
##### Order value: Truncated Normal(m=\\$120, s=\\$20, min=\\$80, max=\\$180). Based on Woolworths’ health report \\$242.58 weekly shop reference [[Woolworths Group, Aug 2025]](https://assets.healthylife.com.au/files/Healthylife-Living-Healthy-Report-2025.pdf?_gl=1*1sa2uj8*_gcl_au*MTA2MzA0MDk2Mi4xNzc5OTMyODQx)
##### Order weight: Order value ÷ $5/kg. Blended estimate across produce, meat, dairy, pantry.
##### Order Assignment: Based on the time windows above, assigned customer in index to different time windows. (0 indexed)

In [7]:
min_weight, max_weight = 80, 180
mean_weight, std = 120, 20
lower_bound, upper_bound = (min_weight - mean_weight) / std, (max_weight - mean_weight) / std
order_values = truncnorm(lower_bound, upper_bound, loc=mean_weight, scale=std).rvs(size=number_of_customers, random_state=random_number_generator)
order_values

array([135.31273851, 117.53990095, 141.66541035, 130.67897886,
        95.95752603, 159.15146842, 134.48516148, 136.11942341,
        99.07962798, 118.10626167, 114.13219365, 149.10212231,
       127.76573522, 138.74485638, 117.76334628, 106.16278519,
       123.22081814,  92.55970079, 139.11774251, 127.12596553,
       134.29174484, 113.29572969, 157.63184633, 144.99964754,
       135.60475748, 104.05810765, 118.90772768,  89.79631774,
       101.17756557, 129.87947183, 133.46233543, 156.75295899,
       111.7901902 , 114.11491152, 119.04660062, 103.70831927,
        99.23187883, 119.34768752, 106.14230618, 129.15431081,
       117.45476803, 139.51136427, 130.84279166, 111.06837555,
       139.47844659, 137.41975978, 114.97913163, 109.74879715,
       129.84889269, 100.03957907, 104.40985884,  82.36499618,
       136.17771789, 128.88549722, 131.12154659, 135.76081535,
       118.52506872, 123.92458579, 100.04316231,  97.89448187])

In [8]:
order_weights = np.round(order_values / 5)
order_weights = np.concatenate(([0.0], order_weights))
order_weights

array([ 0., 27., 24., 28., 26., 19., 32., 27., 27., 20., 24., 23., 30.,
       26., 28., 24., 21., 25., 19., 28., 25., 27., 23., 32., 29., 27.,
       21., 24., 18., 20., 26., 27., 31., 22., 23., 24., 21., 20., 24.,
       21., 26., 23., 28., 26., 22., 28., 27., 23., 22., 26., 20., 21.,
       16., 27., 26., 26., 27., 24., 25., 20., 20.])

In [9]:
boundaries = np.round(np.cumsum(windows_allocation) * number_of_customers).astype(int)
order_counts_per_windows = np.diff(np.concatenate(([0], boundaries))).tolist()
if sum(order_counts_per_windows) != number_of_customers:
    raise ValueError("Not all customer order is distributed across all windows")
order_assignments = np.repeat(range(number_of_windows), order_counts_per_windows)
random_number_generator.shuffle(order_assignments)
order_assignments = np.concatenate(([-1], order_assignments)) #-1 is for depot
customers_by_window = {w: [c for c in customers if int(order_assignments[c]) == w] for w in range(number_of_windows)}
order_assignments

array([-1,  3,  0,  2,  3,  1,  1,  3,  3,  2,  2,  3,  3,  2,  2,  0,  0,
        3,  1,  2,  2,  3,  3,  3,  0,  3,  1,  3,  3,  3,  3,  3,  3,  3,
        2,  3,  2,  3,  0,  2,  1,  3,  0,  2,  2,  3,  3,  3,  3,  3,  2,
        3,  3,  2,  1,  2,  3,  2,  2,  2,  3])

### Vehicle Profiles
##### Van (CBD): Speed is 15 km/hr, labour cost is \\$25.76 per hour, fixed cost is \\$50 and variable cost is \\$1.2 km
##### ebike: Speed is 20 km/hr, labour cost is \\$25.76 per hour, fixed cost is \\$10 and variable cost is \\$0.1 km

In [10]:
van_profile = {
    "type":                 1,
    "capacity_kg":          400.0,
    "speed_kmh":            15.0,
    "staff_cost":           25.76 * total_time / 60,
    "fixed_cost":           50.0,
    "variable_cost_per_km": 1.20,
    "shift_time_in_min":    total_time  #Full 8 hours
}

ebike_profile = {
    "type":                 2,
    "capacity_kg":          30.0,
    "speed_kmh":            20.0,
    "staff_cost":           25.76 * 2,
    "fixed_cost":           10.0,
    "variable_cost_per_km": 0.1,
    "shift_time_in_min":    window_time
}
number_of_vans = 7
max_capacity_in_kg = max(van_profile["capacity_kg"], ebike_profile["capacity_kg"])
fleet: list[dict[str, int | float]] = [van_profile] * number_of_vans
fleet

[{'type': 1,
  'capacity_kg': 400.0,
  'speed_kmh': 15.0,
  'staff_cost': 206.08,
  'fixed_cost': 50.0,
  'variable_cost_per_km': 1.2,
  'shift_time_in_min': 480},
 {'type': 1,
  'capacity_kg': 400.0,
  'speed_kmh': 15.0,
  'staff_cost': 206.08,
  'fixed_cost': 50.0,
  'variable_cost_per_km': 1.2,
  'shift_time_in_min': 480},
 {'type': 1,
  'capacity_kg': 400.0,
  'speed_kmh': 15.0,
  'staff_cost': 206.08,
  'fixed_cost': 50.0,
  'variable_cost_per_km': 1.2,
  'shift_time_in_min': 480},
 {'type': 1,
  'capacity_kg': 400.0,
  'speed_kmh': 15.0,
  'staff_cost': 206.08,
  'fixed_cost': 50.0,
  'variable_cost_per_km': 1.2,
  'shift_time_in_min': 480},
 {'type': 1,
  'capacity_kg': 400.0,
  'speed_kmh': 15.0,
  'staff_cost': 206.08,
  'fixed_cost': 50.0,
  'variable_cost_per_km': 1.2,
  'shift_time_in_min': 480},
 {'type': 1,
  'capacity_kg': 400.0,
  'speed_kmh': 15.0,
  'staff_cost': 206.08,
  'fixed_cost': 50.0,
  'variable_cost_per_km': 1.2,
  'shift_time_in_min': 480},
 {'type': 1,
  '

# Model Setup

### Parameters

#### Big-M
$$
M = 480 + \max_{i \in C}s_i + \frac{\max_{i,j \in N}d_{ij}}{\min_{k \in K}v_k}\times 60
$$

#### Vehicle Capacity
$$
Q = \text{vehicle capacity in kg (homogeneous fleet)}
$$

#### Order Assignment
$$
r_{j} = \text{the assigned delivery time window for customer } j
$$

#### Order Weight
$$
w_j = \text{delivery weight demand of customer } j \text{ in kg}
$$

#### Service Time
$$
s_{i} = \text{service time at node } i
$$

#### Distance between Nodes
$$
d_{ij} = \text{distance between nodes } i \text{ and } j
$$

#### Speed of Vehicles
$$
v_{k} = \text{travel speed of vehicle } k
$$

#### Travel Cost of Vehicles
$$
p_{k} = \text{variable travel cost per km for vehicle } k
$$

#### Staff Cost of Vehicles
$$
L_{k} = \text{staff cost for vehicle } k
$$

#### Fixed Vehicle Cost
$$
F_{k} = \text{fixed cost for vehicle } k
$$

### Decision Variables

In [11]:
model = gp.Model("model")
BIG_M = total_time + max(service_times) + float(store_and_residences_distance_matrix.max().max()) / min(f["speed_kmh"] for f in fleet) * 60.0 #max shift time + max service time + max travel time (max distance / slowest km/h * 60)
arcs = [(i, j) for i in range(total_nodes) for j in range(total_nodes) if i != j]
arc_traveled = model.addVars([(i, j, k) for (i, j) in arcs for k in range(number_of_vans)], vtype=GRB.BINARY, name="arc_traveled")
dispatch = model.addVars(number_of_vans, vtype=GRB.BINARY, name="dispatch")
arrival = model.addVars(total_nodes, lb=0.0, name="arrival")
cumulative_load = model.addVars(total_nodes, ub=max_capacity_in_kg, name="cumulative_load")

Academic license - for non-commercial use only - expires 2027-05-08


# Constraints

## Visiting Constraint
##### Visit each customer only once
$\sum_{i \in N} \sum_{k \in K} x_{i,j,k} = 1 \quad \forall j \in C $

In [12]:
model.addConstrs((arc_traveled.sum("*", c, "*") == 1 for c in customers), name="Visiting Constraint");

### Capacity Constraint
##### For each vehicle, the sum of demands for all customers it visits cannot exceed its capacity in kg
$\sum_{j \in C} w_j \left( \sum_{i \in N} x_{ijk} \right) \leq Q \quad \forall k \in K $

In [13]:
model.addConstrs((gp.quicksum(order_weights[c] * arc_traveled.sum("*", c, v) for c in customers)
                  <= fleet[v]["capacity_kg"] for v in range(number_of_vans)),
                 name="Capacity Constraint");

### Time Windows Constraint
##### Each customer is served within their chosen 2-hours delivery slot (hard constraint — early arrival permitted with waiting; late arrival infeasible).
$ t_{i} \geq a_{r_{i}} \quad \forall i \in C $

$ t_{i} \leq b_{r_{i}} \quad \forall i \in C $

In [14]:
model.addConstrs((arrival[c] >= windows[int(order_assignments[c])][0] for c in customers),
                name="Time Window Constraint Lower Bound")
model.addConstrs((arrival[c] <= windows[int(order_assignments[c])][1] for c in customers),
                name="Time Window Constraint Upper Bound");

### Time Ordering Constraint
##### If a vehicle travels from customer A to customer B, the arrival time at B must be consistent with the departure time from A, i.e., $\text{arrival}_B \geq \text{departure}_A + \text{service time}_A + \text{travel time}_{AB} $
$t_{j} \geq t_{i} + s_{i} + \frac{d_{ij}}{v_k} \times 60 - M(1 - x_{ijk})\quad \forall (i,j)\in A,\ \forall k\in K $

In [15]:
model.addConstrs((arrival[j] >= arrival[i] + service_times[i] + store_and_residences_distance_matrix.iat[i, j] / van_profile["speed_kmh"] * 60
                  - BIG_M * (1 - arc_traveled.sum(i, j, "*")) for (i, j) in arcs if j != 0), name="Time Ordering Constraints");

### Flow Conservation Constraint
##### Every vehicle that arrives at a customer must also depart from that customer
$\sum_{i \in N} x_{ick} = \sum_{j \in N} x_{cjk} \quad \forall c \in C,\ \forall k \in K$

In [16]:
model.addConstrs((arc_traveled.sum("*", c, v) == arc_traveled.sum(c, "*", v) for c in customers
                  for v in range(number_of_vans)),
                  name="Flow Conservation");

### Depot Constraint
##### All vehicles depart from and return to the same store (single depot per instance).
$\sum_{j \in C} x_{0jk} = y_{k} \quad \forall k \in K$

$\sum_{i \in C} x_{i0k} = y_{k} \quad \forall k \in K$

In [17]:
model.addConstrs((arc_traveled.sum(0, "*", v) == dispatch[v] for v in range(number_of_vans)),
                 name="Depot Constraint Out")
model.addConstrs((arc_traveled.sum("*", 0, v) == dispatch[v] for v in range(number_of_vans)),
                 name="Depot Constraint In");

### Subtour elimination
##### Routes must form connected paths from and back to the depot.

$ q_{0} = 0 $

$ q_{i} \geq w_{i} \quad \forall i \in C $

$ q_j \geq q_i + w_j - Q\left(1-\sum_{k\in K}x_{ijk}\right) \quad \forall (i,j)\in A,\ j\neq 0 $

In [18]:
model.addConstr(cumulative_load[0] == 0, name="Depot Initial Load Capacity")
model.addConstrs((cumulative_load[c] >= order_weights[c] for c in customers), name="Minimum Cumulative Load Constraint")
model.addConstrs((cumulative_load[j] >= cumulative_load[i] + order_weights[j]
                  - van_profile["capacity_kg"] * (1 - arc_traveled.sum(i, j, "*")) for (i, j) in arcs if j != 0),
                 name="Cumulative Load Constraint");

### Shift Cap Constraint
##### Each dispatched van must return backs to the depot within its shift time
$t_{i} + s_{i} + \frac{d_{i0}}{v_k} \times 60 \;\leq\; T^{\text{shift}}_{k} + M(1 - x_{i0k}) \quad \forall i \in C,\ \forall k \in K$

In [19]:
model.addConstrs((arrival[c] + service_times[c] + store_and_residences_distance_matrix.iat[c, 0] / fleet[v]["speed_kmh"] * 60 <= fleet[v]["shift_time_in_min"]
       + BIG_M * (1 - arc_traveled[c, 0, v]) for c in customers for v in range(number_of_vans)),
    name="Shift Cap Constraint");

### Model Tuning

In [20]:
# Cut
min_van_needed = int(np.ceil(sum(order_weights) / van_profile["capacity_kg"]))
model.addConstr(gp.quicksum(dispatch[v] for v in range(number_of_vans)) >= min_van_needed, name="Min Van Dispatched")
print(f"min van needed {min_van_needed} ")
print(f"order weights {sum(order_weights)} ")
model.Params.MIPGap = 0.03
model.Params.NoRelHeurWork = 20
model.Params.Presolve = 1
model.Params.Heuristics = 0

Set parameter Heuristics to value 0


# Objective Function

In [21]:
model.setObjective(
    gp.quicksum(dispatch[k] * (fleet[k]["staff_cost"] + fleet[k]["fixed_cost"]) for k in range(number_of_vans))
    + gp.quicksum(fleet[k]["variable_cost_per_km"] * store_and_residences_distance_matrix.iat[i, j] * arc_traveled[i, j, k] for (i, j) in arcs for k in range(number_of_vans)),
    GRB.MINIMIZE,
)
model.optimize()

Best objective 1.093649712000e+03, best bound 1.073079221651e+03, gap 1.8809%


### Solved Results

In [22]:
vans_used = sum(1 for v in range(number_of_vans) if dispatch[v].X > 0.5 and fleet[v]["type"] == 1)
print(f"{vans_used} vans are used with ${model.ObjVal:.2f} cost")

4 vans are used with $1093.65 cost


In [23]:
print_routes(arc_traveled, dispatch, arrival, fleet, total_nodes, store_and_residences_distance_matrix, service_times, order_weights, order_assignments, time_windows_mapping, number_of_windows * window_time)

Vehicle 0 (van) | stops: 16 | load: 379.0 kg | total distance: 13.10 km | total time: 154.4 min | utilization 32.16%
  Route: 0 -> 14 -> 58 -> 57 -> 34 -> 36 -> 50 -> 33 -> 56 -> 30 -> 37 -> 31 -> 7 -> 52 -> 8 -> 45 -> 28 -> 0
    Customer 14 | window 15-17 | arrival 14:00 | from dep  1.94 km | time traveled  7.8 min | weight 28.0 kg
    Customer 58 | window 15-17 | arrival 14:06 | from C14  0.22 km | time traveled  0.9 min | weight 25.0 kg
    Customer 57 | window 15-17 | arrival 14:12 | from C58  0.24 km | time traveled  1.0 min | weight 24.0 kg
    Customer 34 | window 15-17 | arrival 15:28 | from C57  0.31 km | time traveled  1.3 min | weight 23.0 kg
    Customer 36 | window 15-17 | arrival 15:40 | from C34  1.77 km | time traveled  7.1 min | weight 21.0 kg
    Customer 50 | window 15-17 | arrival 16:00 | from C36  1.26 km | time traveled  5.0 min | weight 20.0 kg
    Customer 33 | window 17-19 | arrival 16:06 | from C50  0.22 km | time traveled  0.9 min | weight 22.0 kg
    Custom

In [24]:
display_route_map(arc_traveled, dispatch, fleet, total_nodes, coordinates, route_geometries)

# Extension: Mixed Fleet with Ebikes
Adds ebikes alongside vans. Ebikes have lower fixed and variable costs, but smaller capacity (30 kg), faster speed in CBD and slower speed in suburb vs van  (20 km/hr), and a 2-hour shift cap (part-time in nature). Ebikes are modeled as **multi-trip** because two customer orders (≥16 kg each) cannot fit on one ebike, every ebike trip is a depot→customer→depot round trip. An ebike completes as many trips as fits within its 120-min shift.

### Data

In [25]:
number_of_vans_mixed = vans_used      # match base model's van count for tighter bound given the task is to measure whether ebikes reduce cost
number_of_ebikes_mixed = 20
ebike_reload_minutes = 5
ebike_trip_time = {
    c: (store_and_residences_distance_matrix.iat[0, c] + store_and_residences_distance_matrix.iat[c, 0]) / ebike_profile["speed_kmh"] * 60
       + service_times[c]
    for c in customers
}
min_ebike_trip_time = min(ebike_trip_time[c] for c in customers if order_weights[c] <= ebike_profile["capacity_kg"])
max_trips_per_ebike = int((ebike_profile["shift_time_in_min"] + ebike_reload_minutes) // (min_ebike_trip_time + ebike_reload_minutes))

van_trip_time = store_and_residences_distance_matrix / van_profile["speed_kmh"] * 60

### Warm-start for column generation
The set-partitioning master LP requires an initial pool of feasible routes. We construct that pool
in three layers, in increasing quality:
1. **Singletons** `0 → c → 0` for each customer guarantee LP feasibility, this guarantees every customer
 is visited by some routes in the pool.
2. **Greedy nearest-neighbour** routes built window-by-window add realistic multi-customer
 columns so the first LP relaxation is not pessimistically loose.
3. **Base-MIP routes** ensures the CG solution is at least as good as the
 base model's vans-only optimum.

In [26]:
available_routes = []
traversed = set()

# Ensured all customers are visited
for c in customers:
    total_distance = store_and_residences_distance_matrix.iat[0, c] + store_and_residences_distance_matrix.iat[c, 0]
    total_cost = van_profile["staff_cost"] + van_profile["fixed_cost"] + van_profile["variable_cost_per_km"] * total_distance
    traversed.add(tuple([0, c, 0]))
    available_routes.append({"path": [0, c, 0], "cost": total_cost, "distance": total_distance, "customers": frozenset([c])})

unassigned_customers = set(customers)

# Greedy algorithm to start filling the van from the earliest time windows until the window is exhausted or capacity runs out
while unassigned_customers:
    route = [0]
    load = 0
    current_time = 0
    active_time = 0
    current = 0
    for w in range(number_of_windows):
        # Finding the nearest neighbour in the time windows
        while True:
            chosen_customer = None
            min_travel_time = float("inf")
            for c in customers_by_window[w]:
                if c not in unassigned_customers:
                    continue
                time_to_customer = van_trip_time.iat[current, c]
                return_time = van_trip_time.iat[c, 0]
                window_start_time = windows[int(order_assignments[c])][0]
                window_end_time = windows[int(order_assignments[c])][1]
                arrival = max(current_time + time_to_customer, window_start_time)
                # Can't fit in the window
                if arrival > window_end_time:
                    continue
                # Exceed van capacity
                if load + order_weights[c] > van_profile["capacity_kg"]:
                    continue
                # Exceed shift time
                if active_time + time_to_customer + service_times[c] + return_time > van_profile["shift_time_in_min"]:
                    continue
                if time_to_customer < min_travel_time:
                    chosen_customer = c
                    min_travel_time = time_to_customer
            # Goes to the next time windows if no customer can be fitted in
            if chosen_customer is None:
                break
            time_to_customer = van_trip_time.iat[current, chosen_customer]
            arrival_at_chosen = max(current_time + time_to_customer, windows[int(order_assignments[chosen_customer])][0])
            current_time = arrival_at_chosen + service_times[chosen_customer]
            active_time += time_to_customer + service_times[chosen_customer]
            load += order_weights[chosen_customer]
            route.append(chosen_customer)
            current = chosen_customer
            unassigned_customers.remove(chosen_customer)
    if len(route) > 1:
        route.append(0)
        if tuple(route) not in traversed:
            traversed.add(tuple(route))
            total_distance = sum(store_and_residences_distance_matrix.iat[route[k], route[k + 1]] for k in range(len(route) - 1))
            total_cost = van_profile["staff_cost"] + van_profile["fixed_cost"] + van_profile["variable_cost_per_km"] * total_distance
            available_routes.append({"path": route,
                                     "cost": total_cost,
                                     "distance": total_distance,
                                     "customers": frozenset(route[1:-1])})

if model.SolCount > 0:
    for v in range(number_of_vans):
        if dispatch[v].X > 0.5:
            route = reconstruct_route(arc_traveled, total_nodes, v)
            total_distance = sum(store_and_residences_distance_matrix.iat[route[k], route[k + 1]] for k in range(len(route) - 1))
            total_cost = van_profile["staff_cost"] + van_profile["fixed_cost"] + van_profile["variable_cost_per_km"] * total_distance
            if len(route) > 2 and tuple(route) not in traversed:
                traversed.add(tuple(route))
                available_routes.append({"path": route,
                                         "cost": total_cost,
                                         "distance": total_distance,
                                         "customers": frozenset(route[1:-1])})

  # Master Setup
  The set-partitioning master picks (a) van routes from the column generated pool and (b) per ebike depot→customer→depot round trips.
  Solved as an LP during column generation to extract duals, then as an IP over the stabilised pool.

  ### Decision Variables
  #### Route Selection Variable
  $$
  \lambda_r =
  \begin{cases}
  1, & \text{if van route } r \text{ is selected from the pool} \\
  0, & \text{otherwise}
  \end{cases}
  $$

  #### Ebike Dispatch Variable
  $$
  y_b =
  \begin{cases}
  1, & \text{if ebike } b \text{ is dispatched} \\
  0, & \text{otherwise}
  \end{cases}
  $$

  #### Ebike Window Assignment Variable
  $$
  z_{b,w} =
  \begin{cases}
  1, & \text{if ebike } b \text{ is assigned to window } w \\
  0, & \text{otherwise}
  \end{cases}
  $$

  #### Ebike Service Variable
  $$
  u_{b,c} =
  \begin{cases}
  1, & \text{if ebike } b \text{ serves customer } c \text{ (round trip depot}\to c \to \text{depot)} \\
  0, & \text{otherwise}
  \end{cases}
  $$

  ### Sets
  #### Route Pool
  $$
  R = \text{set of feasible van routes generated by warm-start + pricing}
  $$

  #### Ebike Set
  $$
  B = \text{set of available ebikes}
  $$

  #### Customer Set
  $$
  C = \text{set of all customers (reused from base model)}
  $$

  #### Time Windows
  $$ T = \{1,2,3,4\}$$
  $$ \text{ each } h \in T \text{ corresponds to a delivery time window with lower and upper bounds }[a_h,\ b_h]   $$
  $$   a_{h} = \text{start time of delivery window } h  $$
  $$ b_{h} = \text{end time of delivery window } h   $$


  ### Parameters
  #### Route Cost
  $$
  c_r = L^{\text{van}} + F^{\text{van}} + p^{\text{van}} \sum_{(i,j) \in r} d_{i,j}
  \qquad \text{total cost of route } r \text{ in the pool}
  $$

  #### Van Fleet Size
  $$
  N_{\text{vans}} = \text{number of vans physically available}
  $$

  #### Ebike Capacity
  $$
  Q^{\text{ebike}} = \text{ebike capacity in kg}
  $$

  #### Ebike Shift Length
  $$
  T^{\text{shift}}_{\text{ebike}} = \text{ebike shift cap in minutes}
  $$

  #### Ebike Round-Trip Time
  $$
  t^{\text{trip}}_c = \frac{60}{v^{\text{ebike}}}(d_{0,c} + d_{c,0}) + s_c
  \qquad \text{drive + service time per ebike trip (excludes reload)}
  $$

  #### Ebike Trip-Count Cap
  $$
  M^{\text{trips}} = \left\lfloor \frac{T^{\text{shift}}_{\text{ebike}} + \tau^{\text{reload}}}{\min_{c \in C} t^{\text{trip}}_c + \tau^{\text{reload}}} \right\rfloor
  $$

  #### Ebike Staff Cost
  $$
  L^{\text{ebike}} = \text{ebike staff cost per dispatch}
  $$

  #### Ebike Fixed Cost
  $$
  F^{\text{ebike}} = \text{ebike fixed cost per dispatch}
  $$

  #### Ebike Variable Cost
  $$
  p^{\text{ebike}} = \text{ebike variable cost per km}
  $$

  (Customer-level parameters $w_c$, $s_c$, $r_c$ and arc distance $d_{i,j}$ are reused from the base model.)

  # Master Constraints

  ## Visiting Constraint
  ##### Each customer is served exactly once either by van or ebike.
  $$
  \sum_{r \in R\,:\, c \in r} \lambda_r \;+\; \sum_{b \in B} u_{b,c} \;=\; 1 \qquad \forall c \in C
  $$

  ## Van Count Constraint
  ##### Total van routes selected can't exceed the available van fleet.
  $$
  \sum_{r \in R} \lambda_r \;\le\; N_{\text{vans}}
  $$

  ## Ebike Single Window Constraint
  ##### A dispatched ebike picks exactly one window.
  $$
  \sum_{w \in W} z_{b,w} \;=\; y_b \qquad \forall b \in B
  $$

  ## Ebike Time Window Constraint
  ##### Ebike $b$ can serve customer $c$ only if $b$'s assigned window equals $c$'s window.
  $$
  u_{b,c} \;\le\; z_{b,\,r_c} \qquad \forall b \in B,\ c \in C
  $$

  ## Ebike Shift Constraint
  ##### Drive + service time across served customers, plus reload time between trips ($K$ trips on a dispatched ebike incur $K-1$ inter-trip reloads), cannot exceed the shift.
  $$
  \sum_{c \in C} t^{\text{trip}}_c \cdot u_{b,c}
  \;+\; \tau^{\text{reload}} \Bigl( \sum_{c \in C} u_{b,c} - y_b \Bigr)
  \;\le\; T^{\text{shift}}_{\text{ebike}} \cdot y_b
  \qquad \forall b \in B
  $$

  ## Ebike Trips per Window
  ##### Within any single window, an ebike can't serve more customers than the trip-count cap. This is implied by Ebike Time Window Constraint and Ebike Trip Count Cap. This is only added to tighten the bound for LP relaxation.
  $$
  \sum_{\substack{c \in C \\ r_c = w}} u_{b,c} \;\le\; M^{\text{trips}} \cdot z_{b,w} \qquad \forall b \in B,\ w \in W
  $$

  ## Ebike Symmetry Breaking
  ##### Force a canonical dispatch order to eliminate equivalent permutations of ebikes.
  $$
  y_b \;\ge\; y_{b+1} \qquad \forall b = 0, \ldots, |B|-2
  $$

  ## Heavy-Customer Exclusion
  ##### Customers heavier than ebike capacity can never be served by ebike.
  $$
  u_{b,c} = 0 \qquad \forall b \in B,\ c \in C \text{ such that } w_c > Q^{\text{ebike}}
  $$

  # Master Objective Function
  Minimise total cost: selected van routes + ebike dispatch + ebike variable km.
  $$
  \min \;\;
  \underbrace{\sum_{r \in R} c_r \, \lambda_r}_{\text{(1) van routes}}
  \;+\; \underbrace{\big(L^{\text{ebike}} + F^{\text{ebike}}\big) \sum_{b \in B} y_b}_{\text{(2) ebike dispatch}}
  \;+\; \underbrace{p^{\text{ebike}} \sum_{b \in B} \sum_{c \in C} \big(d_{0,c} + d_{c,0}\big) \, u_{b,c}}_{\text{(3) ebike round-trip
  km}}
  $$

  #### Duals consumed by pricing
  The LP relaxation yields two dual vectors used by the pricing subproblem to compute reduced cost:
  - $\pi_c$ from the Visiting Constraint (one per customer; free-sign).
  - $\pi_v$ from the Van Count Constraint (one scalar; $\le 0$ when binding).

  A candidate van route $r$ has reduced cost $\bar c_r = c_r - \sum_{c \in r} \pi_c - \pi_v$, and is worth adding to the pool iff $\bar
  c_r < 0$.

In [27]:
def master(
    available_routes: list[dict],
    customers: range,
    number_of_windows: int,
    number_of_vans: int,
    number_of_ebikes: int,
    order_assignments: np.ndarray,
    order_weights: np.ndarray,
    ebike_profile: dict[str, int | float],
    ebike_trip_time: dict[int, float],
    ebike_reload_minutes: int,
    max_trips_per_ebike: int,
    distance_matrix: pd.DataFrame,
    is_integer: bool) -> tuple[gp.Model, gp.tupledict, gp.tupledict, gp.tupledict, gp.tupledict, gp.tupledict, gp.Constr]:


      # Build the set-partitioning master for mixed-fleet
      mixed_model = gp.Model("master")
      vtype = GRB.BINARY if is_integer else GRB.CONTINUOUS
      ebike_idx = range(number_of_ebikes)

      # Decision variables
      # lambda_r[r] : pick (1) or skip (0) route r from the available pool
      lambda_r = mixed_model.addVars(len(available_routes), vtype=vtype, lb=0.0, ub=1.0, name="lambda")

      # ebike_dispatch[b] : 1 if ebike b is dispatched
      ebike_dispatch = mixed_model.addVars(ebike_idx, vtype=vtype, name="ebike_dispatch")

      # ebike_windows[b, w] : 1 if dispatched ebike b is assigned to window w
      ebike_windows = mixed_model.addVars([(b, w) for b in ebike_idx for w in range(number_of_windows)], vtype=vtype, name="ebike_window")

      # ebike_serviced[b, c] : 1 if ebike b serves customer c (depot→c→depot)
      ebike_serviced = mixed_model.addVars([(b, c) for b in ebike_idx for c in customers], vtype=vtype, name="ebike_serviced")

      # Constraints
      # Visiting Constraint
      # Each customer is visited once by some selected van route or by exactly one ebike's round trip
      visiting_constraints = mixed_model.addConstrs((gp.quicksum(lambda_r[r] for r, rr in enumerate(available_routes) if c in rr["customers"])
           + gp.quicksum(ebike_serviced[b, c] for b in ebike_idx) == 1 for c in customers), name="Visiting Constraint")

      # Van Count Constraint
      # Total van routes selected ≤ number of vans physically available.
      van_counts_constraints = mixed_model.addConstr(gp.quicksum(lambda_r[r] for r in range(len(available_routes))) <= number_of_vans, name="Van Counts Constraint")

      # Ebike Single Window Constraint
      # A dispatched ebike is only in one window.
      mixed_model.addConstrs((gp.quicksum(ebike_windows[b, w] for w in range(number_of_windows)) == ebike_dispatch[b] for b in ebike_idx), name="Ebike Single Window Constraint")

      # Ebike Time Window Constraint
      # ebike b can serve customer c only if b's chosen window equals c's window
      mixed_model.addConstrs((ebike_serviced[b, c] <= ebike_windows[b, int(order_assignments[c])] for b in ebike_idx for c in customers), name="Ebike Time Window Constraint")

      # Ebike Shift Constraint
      # Total round-trip minutes across all served customers within the ebike's shift cap (RHS = 0 if not dispatched)
      mixed_model.addConstrs((gp.quicksum(ebike_serviced[b, c] * ebike_trip_time[c] for c in customers) + ebike_reload_minutes * (gp.quicksum(ebike_serviced[b, c] for c in customers) - ebike_dispatch[b]) <= ebike_profile["shift_time_in_min"] * ebike_dispatch[b] for b in ebike_idx), name="Ebike Shift Constraint")

      # Tuning
      # Heavy customer exclusion: orders > ebike capacity can NEVER be ebike-served
      for b in ebike_idx:
          for c in customers:
              if order_weights[c] > ebike_profile["capacity_kg"]:
                  ebike_serviced[b, c].UB = 0

      # Trip count cap: at most max_trips_per_ebike customers per dispatch
      mixed_model.addConstrs((gp.quicksum(ebike_serviced[b, c] for c in customers) <= max_trips_per_ebike * ebike_dispatch[b] for b in ebike_idx), name="ebike_max_trips")

      # Ebike per window trip: maximum trips an ebike can take in that time windows
      mixed_model.addConstrs((gp.quicksum(ebike_serviced[b, c] for c in customers if int(order_assignments[c]) == w) <= max_trips_per_ebike * ebike_windows[b, w]
                              for b in ebike_idx for w in range(number_of_windows)), name="ebike_trips_per_window")

      # Ebike symmetry breaking
      # Force dispatch ordering: ebike 0 before ebike 1 before ebike 2... This eliminates equivalent ebike permutations
      mixed_model.addConstrs((ebike_dispatch[b] >= ebike_dispatch[b + 1] for b in range(number_of_ebikes - 1)), name="ebike_symmetry")

      # Objective
      # Minimize total cost:
      #   1) Sum (van route cost) × lambda_r: selected van routes
      #   2) Sum (ebike fixed + staff)      : for each dispatched ebike
      #   3) Sum ebike variable km cost     : per ebike round trip
      mixed_model.setObjective(
          gp.quicksum(available_routes[r]["cost"] * lambda_r[r]
                      for r in range(len(available_routes)))
          + gp.quicksum((ebike_profile["staff_cost"] + ebike_profile["fixed_cost"]) * ebike_dispatch[b] for b in ebike_idx)
          + gp.quicksum(ebike_profile["variable_cost_per_km"] * (distance_matrix.iat[0, c] + distance_matrix.iat[c, 0]) * ebike_serviced[b, c]
                        for b in ebike_idx for c in customers), GRB.MINIMIZE)

      return mixed_model, lambda_r, ebike_dispatch, ebike_windows, ebike_serviced, visiting_constraints, van_counts_constraints

### Pricing Subproblem
Each iteration the master LP returns dual values, and the pricer's job is
to find new van routes whose **reduced cost** is negative.
For a van route $r = (0, c_1, c_2, \dots, c_k, 0)$ the reduced cost is

$$
\bar{c}_r \;=\; \underbrace{\text{staff} + \text{fixed} + v \cdot d_r}_{\text{route cost}}
          \;-\; \sum_{c \in r} \pi_c \;-\; \mu
$$

where $\pi_c$ is the dual on customer $c$'s visiting constraint and $\mu$ is the
dual on the van-count constraint. A route is useful to the master iff
$\bar{c}_r < 0$.

Because enumerating every feasible route is intractable, the pricer is a
**heuristic depth-first search** with three feasibility filters
(time window, capacity, shift length) and a greedy branching rule.

#### Algorithm outline

1. **Initial**: for each customer $c$, sorted by descending dual $\pi_c$ (most
 "valuable" customer first), start a partial route [0, c].
2. **Extend** with DFS:
 - At each node, compute every feasible next customer.
 - Score each candidate by its incremental reduced cost contribution
   $v \cdot d_{\text{cur},c} - \pi_c$ (lower = more promising).
 - Recurse on the best 15 candidates only
3. **Close** at every prefix: `try_end` tentatively appends a return-to-depot
 arc and tests whether the resulting full route has negative reduced cost.
 If yes, and the route isn't already in the pool, save it.
4. **Stop** when either `max_routes` columns are found or `time_limit` seconds
 pass — whichever comes first.

#### Limitations
This pricer is a **truncated depth-first search** influenced by the Pulse
algorithm ([Lozano, Duque & Medaglia, 2015](https://doi.org/10.1287/trsc.2014.0582)),
- **Branch-factor truncation:** Extensions ranked below the top 15 are silently
discarded. The single arc that would lead to the most negative reduced cost
can sit at rank 16 and be lost forever.
- **Time budget:** The DFS aborts when the wall-clock cap is hit, even mid-tree.



In [28]:
def price(duals: dict[int, float],
        dual_vehicle: float,
        customers: range,
        windows: list[tuple[int, int]],
        order_assignments: np.ndarray,
        order_weights: np.ndarray,
        service_times: list[int],
        van_profile: dict[str, int | float],
        van_trip_time: pd.DataFrame,
        distance_matrix: pd.DataFrame,
        traversed: set[tuple[int, ...]]) -> list[dict]:
    time_limit = 60
    max_routes = 200
    branch_factor = 15
    eps = -1e-6
    found, found_keys = [], set()
    start = time.time()
    def try_end(prefix: list[int], dist_so_far: float) -> None:
        if len(prefix) < 2:
            return
        full = prefix + [0]
        full_dist = dist_so_far + distance_matrix.iat[prefix[-1], 0]
        full_cost = van_profile["staff_cost"] + van_profile["fixed_cost"] + van_profile["variable_cost_per_km"] * full_dist
        visited_customers = frozenset(prefix[1:])
        reduced = full_cost - sum(duals[c] for c in visited_customers) - dual_vehicle
        if reduced < eps:
            key = tuple(full)
            if key not in found_keys and key not in traversed:
                found_keys.add(key)
                found.append({"path": full,
                              "cost": full_cost,
                              "distance": full_dist,
                              "customers": visited_customers,
                              "reduced": reduced})

    def dfs(prefix: list[int], t: float, load: float, dist_so_far: float, active_time: float) -> None:
        if len(found) >= max_routes or time.time() - start > time_limit:
            return
        try_end(prefix, dist_so_far)
        current = prefix[-1]
        candidates = []
        prefix_set = set(prefix)
        for c in customers:
            if c in prefix_set:
                continue
            time_to_customer = van_trip_time.iat[current, c]
            return_time = van_trip_time.iat[c, 0]
            window_start_time = windows[int(order_assignments[c])][0]
            window_end_time = windows[int(order_assignments[c])][1]
            arrival = max(t + time_to_customer, window_start_time)
            # Can't fit in the window
            if arrival > window_end_time:
                continue
            # Exceed van capacity
            if load + order_weights[c] > van_profile["capacity_kg"]:
                continue
            # Exceed shift time
            if active_time + time_to_customer + service_times[c] + return_time > van_profile["shift_time_in_min"]:
                continue
            # Reduced cost
            c_bar = van_profile["variable_cost_per_km"] * distance_matrix.iat[current, c] - duals[c]
            candidates.append((c_bar, c, arrival, time_to_customer))
        candidates.sort()
        for _, c, arr, travel in candidates[:branch_factor]:
            dfs(prefix + [c], arr + service_times[c],
                load + order_weights[c],
                dist_so_far + distance_matrix.iat[current, c],
                active_time + travel + service_times[c])

    for customer in sorted(customers, key=lambda c: -duals[c]):
        if len(found) >= max_routes or time.time() - start > time_limit:
            break
        trip_time = van_trip_time.iat[0, customer]
        arrival = max(trip_time, windows[int(order_assignments[customer])][0])
        dfs([0, customer], arrival + service_times[customer], order_weights[customer],
            distance_matrix.iat[0, customer],
            trip_time + service_times[customer])
    return found

### Column Generation Loop
Alternates between solving the master LP and calling the pricing subproblem until no improving route can be found.

##### Each iteration performs four steps:
1. Build & solve the master LP with the current `available_routes` pool.
2. Extract duals from the Visiting Constraint via `visit[c].Pi` and from the Van Count Constraint via `van_count.Pi`.
3. Call the pricing subproblem to search for van routes with reduced cost $\bar c_r = c_r -
\sum_{c \in r} \pi_c - \pi_v < 0$.
4. Add the returned result to `available_routes`. If pricing returns an empty list, then the loop exits.

In [29]:
# Using methods with long parameters instead of one single class is to preserve the illustrative nature of explaining each part with the notebook
for i in range(30):
    m, _, _, _, _, visit, van_count = master(available_routes,
                                             customers,
                                             number_of_windows,
                                             number_of_vans_mixed,
                                             number_of_ebikes_mixed,
                                             order_assignments,
                                             order_weights,
                                             ebike_profile,
                                             ebike_trip_time,
                                             ebike_reload_minutes,
                                             max_trips_per_ebike,
                                             store_and_residences_distance_matrix,
                                             False)
    t0 = time.time()
    m.Params.OutputFlag = 0
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        print(f"iter {i}: LP status {m.Status}; stopping")
        break
    duals = {c: visit[c].Pi for c in customers}
    dual_vehicle = van_count.Pi
    print(f"iter {i}: routes={len(available_routes):2d} Obj=${m.ObjVal:.2f} vehicle_dual={dual_vehicle:.2f}  "
          f"run_time={time.time()-t0:.1f}s")
    t0 = time.time()
    new_routes = price(duals,
                     dual_vehicle,
                     customers,
                     windows,
                     order_assignments,
                     order_weights,
                     service_times,
                     van_profile,
                     van_trip_time,
                     store_and_residences_distance_matrix,
                     traversed)
    if not new_routes:
        print(f"Converged: {len(available_routes)} routes, Obj=${m.ObjVal:.2f}")
        break
    print(f"        +{len(new_routes)} new routes min reduced cost={min(r['reduced'] for r in new_routes):.3f} run time={time.time()-t0:.1f}s")
    for r in new_routes:
        traversed.add(tuple(r["path"]))
        available_routes.append({"path": r["path"],
                               "cost": r["cost"],
                               "distance": r["distance"],
                               "customers": r["customers"]})

Converged: 1669 routes, Obj=$858.18


In [30]:
root_master_lp_value = m.ObjVal
m_final, lambda_final, ebike_dispatch_final, ebike_window_final, ebike_serves_final, _, _ = master(available_routes,
                                                                                                   customers,
                                                                                                   number_of_windows,
                                                                                                   number_of_vans_mixed,
                                                                                                   number_of_ebikes_mixed,
                                                                                                   order_assignments,
                                                                                                   order_weights,
                                                                                                   ebike_profile,
                                                                                                   ebike_trip_time,
                                                                                                   ebike_reload_minutes,
                                                                                                   max_trips_per_ebike,
                                                                                                   store_and_residences_distance_matrix,
                                                                                                   True)
m_final.optimize()

if m_final.SolCount > 0:
    van_route_selected = [(r, available_routes[r]) for r in range(len(available_routes)) if lambda_final[r].X > 0.5]
    ebikes_used = [b for b in range(number_of_ebikes_mixed) if ebike_dispatch_final[b].X > 0.5]
    print(f"\nCost: ${m_final.ObjVal:.2f}  |  vans used: {len(van_route_selected)}  |  ebikes used: {len(ebikes_used)}")
    print(f"Root master LP bound from CG: ${root_master_lp_value:.2f}")
    print(f"Integrality gap (LP-IP): {(m_final.ObjVal - root_master_lp_value) / m_final.ObjVal * 100:.2f}%")
    print(f"Solver MIPGap at termination: {m_final.MIPGap * 100:.2f}% ")
    print("\nVan routes:")
    for r, route in van_route_selected:
        customer_served = sorted(route["customers"])
        total_trip_time = sum(service_times[c] for c in customer_served) + route["distance"] / van_profile["speed_kmh"] * 60
        utilization = total_trip_time/van_profile["shift_time_in_min"] * 100
        print(f"  ${route['cost']:.2f}  {' -> '.join(map(str, route['path']))}")
        print(f"  Utilization: {utilization:.2f}%")
    print("\nEbike trips:")
    for b in ebikes_used:
        served = sorted(c for c in customers if ebike_serves_final[b, c].X > 0.5)
        chosen_w = [w for w in range(number_of_windows) if ebike_window_final[b, w].X > 0.5]
        total_trip_time = sum(ebike_trip_time[c] for c in served)
        utilization = total_trip_time/ebike_profile["shift_time_in_min"] * 100
        print(f"  Ebike {b}  window {chosen_w[0] if chosen_w else '?'}  serves {served}")
        print(f"  Utilization: {utilization:.2f}%")
else:
    print(f"IP solve returned no solution (status={m_final.Status})")


Cost: $908.52  |  vans used: 1  |  ebikes used: 10
Root master LP bound from CG: $858.18
Integrality gap (LP-IP): 5.54%
Solver MIPGap at termination: 0.00% 

Van routes:
  $274.19  0 -> 18 -> 40 -> 26 -> 6 -> 54 -> 5 -> 55 -> 9 -> 43 -> 19 -> 32 -> 47 -> 1 -> 23 -> 49 -> 0
  Utilization: 38.62%

Ebike trips:
  Ebike 0  window 3  serves [17, 30, 33, 46, 60]
  Utilization: 82.71%
  Ebike 1  window 3  serves [8, 27, 29, 35, 41]
  Utilization: 83.24%
  Ebike 2  window 3  serves [4, 7, 25, 28, 45]
  Utilization: 83.26%
  Ebike 3  window 2  serves [36, 44, 50, 57, 58]
  Utilization: 78.04%
  Ebike 4  window 2  serves [3, 20, 34, 39]
  Utilization: 73.54%
  Ebike 5  window 3  serves [11, 31, 37, 52, 56]
  Utilization: 81.79%
  Ebike 6  window 0  serves [15, 16, 24]
  Utilization: 77.21%
  Ebike 7  window 0  serves [2, 38, 42]
  Utilization: 50.84%
  Ebike 8  window 3  serves [12, 21, 22, 48, 51]
  Utilization: 81.40%
  Ebike 9  window 2  serves [10, 13, 14, 53, 59]
  Utilization: 76.91%


In [31]:
display_mixed_route_map(available_routes,
                      lambda_final,
                      ebike_dispatch_final,
                      ebike_serves_final,
                      number_of_ebikes_mixed,
                      customers,
                      coordinates,
                      route_geometries)